# 30 · Skin data description for the Methods (review round 3, item 6.3)

Reads the skin manifests and writes **aggregate counts only** (no image, lesion or patient identifiers): images, lesions and patients before and after the analysis exclusions, diagnosis counts, and the distribution of every available demographic, site, device, image-type, collection and date field. Categories with fewer than 10 images are merged into "other (each under 10)". Runtime: under a minute, CPU.

In [1]:
from google.colab import drive
drive.mount("/content/drive")
import os, re, glob, json, datetime
import numpy as np, pandas as pd
DERM = "/content/drive/MyDrive/stochastic_orthogonality/data/derm"
OUT = "/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2/04_results/revisions_c2/skin_data_description_c3.json"
CLASSES = ["MEL", "BCC", "SCC", "AK", "NV", "BKL", "DF", "VASC"]
files = sorted(glob.glob(f"{DERM}/**/*manifest*.csv", recursive=True)) + sorted(glob.glob(f"{DERM}/**/*metadata*.csv", recursive=True))
print("\n".join(os.path.relpath(f, DERM) for f in files))

Mounted at /content/drive
manifest_BCN20000.csv
manifest_HAM10000.csv
manifest_HIBA.csv
manifest_PAD.csv


In [2]:
ID_LIKE = re.compile(r"(^|_)(id|isic_id|image|image_id|lesion_id|patient_id|path|file|filename|sha|hash|url)($|_)", re.I)
KEEP = re.compile(r"age|sex|gender|site|location|anatom|image_type|modality|device|camera|dermoscop|attribution|collection|licen|copyright|date|created|year|fitz|skin_type|split|source|dataset|diagnosis_confirm|confirm|biops|histo", re.I)
def small_merge(vc):
    big = vc[vc >= 10]; rest = int(vc[vc < 10].sum())
    out = {str(k): int(v) for k, v in big.items()}
    if rest: out["other (each under 10)"] = rest
    return out
def describe(path):
    d = pd.read_csv(path, low_memory=False); r = {"file": os.path.relpath(path, DERM), "columns": list(d.columns), "rows": len(d),
         "file_modified_utc": datetime.datetime.utcfromtimestamp(os.path.getmtime(path)).isoformat(timespec="minutes")}
    for col, key in (("lesion_id", "lesions"), ("patient_id", "patients")):
        if col in d.columns: r[key] = int(d[col].nunique())
    if "dx" in d.columns:
        keep = d.dx.isin(CLASSES)
        if "raw_sha256" in d.columns: keep &= ~d.duplicated("raw_sha256", keep="first")
        e = d[keep]; r["after_exclusions"] = {"images": int(len(e)), **({"lesions": int(e.lesion_id.nunique())} if "lesion_id" in e else {}),
                                              **({"patients": int(e.patient_id.nunique())} if "patient_id" in e else {}), "dx": small_merge(e.dx.value_counts())}
        r["excluded"] = {"not_in_eight_classes": int((~d.dx.isin(CLASSES)).sum()), **({"byte_duplicates": int(d.duplicated("raw_sha256", keep="first").sum())} if "raw_sha256" in d else {})}
        d = e
    fields = {}
    for c in d.columns:
        if ID_LIKE.search(c) or not KEEP.search(c): continue
        s = d[c]
        if re.search(r"date|created", c, re.I):
            t = pd.to_datetime(s, errors="coerce"); fields[c] = {"min": str(t.min())[:10], "max": str(t.max())[:10], "missing": int(t.isna().sum())}
        elif pd.api.types.is_numeric_dtype(s) and s.nunique() > 15:
            fields[c] = {"median": float(s.median()), "q1": float(s.quantile(0.25)), "q3": float(s.quantile(0.75)), "missing": int(s.isna().sum())}
        else:
            fields[c] = {**small_merge(s.astype(str).replace("nan", "missing").value_counts())}
    r["fields"] = fields
    return r
REC = {"written_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), "note": "aggregate counts only; cells under 10 merged", "files": [describe(f) for f in files]}
json.dump(REC, open(OUT, "w"), indent=1, default=str)
for r in REC["files"]:
    print("\n==", r["file"], "| rows", r["rows"], "| modified", r["file_modified_utc"])
    for k in ("lesions", "patients", "excluded", "after_exclusions"):
        if k in r: print(" ", k, r[k])
    for c, v in r["fields"].items(): print("   ", c, v)
print("\nwritten", OUT)

/tmp/ipykernel_818/414749829.py:10: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  "file_modified_utc": datetime.datetime.utcfromtimestamp(os.path.getmtime(path)).isoformat(timespec="minutes")}
/tmp/ipykernel_818/414749829.py:10: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  "file_modified_utc": datetime.datetime.utcfromtimestamp(os.path.getmtime(path)).isoformat(timespec="minutes")}
/tmp/ipykernel_818/414749829.py:10: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.from


== manifest_BCN20000.csv | rows 18946 | modified 2026-09-26T20:15
  lesions 5440
  patients 0
  excluded {'not_in_eight_classes': 1470, 'byte_duplicates': 0}
  after_exclusions {'images': 17476, 'lesions': 4981, 'patients': 0, 'dx': {'NV': 5647, 'MEL': 4636, 'BCC': 3676, 'BKL': 1551, 'AK': 1088, 'SCC': 559, 'DF': 168, 'VASC': 151}}
    source {'BCN20000': 17476}
    modality {'dermoscopy': 17476}
    license {'CC-BY-NC': 17476}

== manifest_HAM10000.csv | rows 11720 | modified 2026-09-26T20:15
  lesions 8838
  patients 0
  excluded {'not_in_eight_classes': 0, 'byte_duplicates': 0}
  after_exclusions {'images': 11720, 'lesions': 8838, 'patients': 0, 'dx': {'NV': 7737, 'BKL': 1338, 'MEL': 1305, 'BCC': 622, 'SCC': 229, 'VASC': 180, 'DF': 160, 'AK': 149}}
    source {'HAM10000': 11720}
    modality {'dermoscopy': 11720}
    license {'CC-BY-NC': 11720}

== manifest_HIBA.csv | rows 1616 | modified 2026-09-26T20:15
  lesions 1246
  patients 623
  excluded {'not_in_eight_classes': 0, 'byte_du

**Also please paste here, if you have them:** the ISIC download date(s) for HAM10000, BCN20000 and HIBA, and the ISIC collection names or numbers shown on the archive page you downloaded from. These are not in the manifests unless the download saved them.